# 🎯 conviction_flow — informed_flow, made selective and fundamentals-gated

A **copy of `informed_flow`** — same alt-data composite (insider buying + rating
changes + price-target revisions + short-interest avoidance), same eligibility, same
TopK-Drop hysteresis selector, same portfolio-mode book. It exists because on a ~100-
name universe with ~10 slots, **far more names clear `q_in` than there are slots** —
most qualifiers are turned away (`slot_blocked`). Raising `q_in` narrows on the *same*
axis; this strategy narrows on **orthogonal** axes and rotates faster into *decisively*
better names, via five levers — **all OFF by default, so at defaults it reproduces
`informed_flow` exactly**:

- **Fundamentals quality gate** (`min_quality_pct`) — a name may open only if it clears
  a cross-sectional fundamentals-quality floor (book-to-price + earnings yield + gross
  profitability, reusing `value_factor`'s SEC-XBRL factors). A veto *on top of* the flow.
- **Signal-agreement gate** (`min_agree`) — require N individual flow sleeves to each
  rank the name top-quantile, not just the blended composite.
- **Conviction bar** (`min_score`) — require the composite score *level* to clear a floor.
- **Margin displacement** (`displace` + `displace_margin`) — rotate into a challenger only
  when it beats the weakest holding by a margin (close faster on a decisive edge).
- **Rank-decay exit** (`exit_decay`) — sell a holding faded far from its peak-while-held.

Judge it against **`informed_flow` itself**, not only the buy-and-hold baseline
(`notebooks/04_compare.ipynb`): selectivity that does not beat the parent out of sample
is not worth its complexity.

> Data must be loaded first: run `notebooks/03_altdata_update.ipynb` (needs the alt-data
> **and** `--dataset fundamentals` for the quality gate).


## 1 · Load the panel and attach alt-data features

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from dataclasses import replace
from scipy import stats

from trading_simple.config import load_settings
from trading_simple.db import make_engine
from trading_simple import backtest as bt, features, jobs, optimize as opt, plots, strategies
from trading_simple.data import load_panel, symbols_with_bars

settings = load_settings()
engine = make_engine(settings)
strat = strategies.load("conviction_flow")
d = strat.data

syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]), d["bar"],
                         d["start"], d.get("end"), limit=d.get("max_symbols"))
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
features.attach_features(panel, engine, d["features"], settings=settings)

print(f"{d['universe']}: {len(syms)} symbols, {panel.close.shape[0]} daily bars "
      f"{panel.close.index[0].date()} → {panel.close.index[-1].date()}")
for name, frame in panel.features.items():
    active = int((frame.abs() > 0).any().sum())
    print(f"  feature {name:15s}: {active}/{len(syms)} symbols with data")

## 2 · Strategy overview

The parameter table below is rendered straight from `config.yaml`, so it never
drifts from what actually runs.

In [ ]:
print("defaults (params):")
for k, v in strat.params.items():
    print(f"  {k:16s} = {v}")
print("\nsweep grid:")
for k, v in strat.sweep.items():
    print(f"  {k:16s} : {v}")
print("\noptuna search space:")
for k, v in strat.optuna["space"].items():
    print(f"  {k:16s} : {v}")

## 3 · One backtest at the defaults

Costs are always on (commission + half-spread + slippage per leg). Read
`pct_profitable` (share of names that made money) as the breadth check — a
cross-sectional strategy should win across many names, not one lucky ticker.

In [ ]:
res = bt.run_strategy(strat, panel, settings=settings)
stats0 = bt.portfolio_stats(res)
for k, v in stats0.items():
    print(f"  {k:16s} = {v:.4f}")
plots.plot_equity(bt.equity_curve(res)).show()

## 4 · Why did this name enter (or leave) the bucket?

Entries are not a threshold on anything you can see on a price chart. A name is
bought when its **cross-sectional percentile rank** crosses `q_in`, and sold when
that rank falls back through `q_out`. So "why is AAPL in the book?" has no answer
until you can see the rank, the bands, and what moved it.

`strategy.explain(panel, symbol, **params)` returns that audit trail. It shares
its whole computation with `signals()` (both call `_compose`), so **the picture
below cannot drift from what the backtest actually traded** — a test asserts they
agree.

Three things it makes visible, each of which looks like a bug until you see it:

1. **A name can cross with no event of its own.** The rank is *relative*: a name
   rises when its peers fall. The `own` / `peer` columns split every bar's rank
   change into exactly those two causes (they sum to the change with no residual),
   and `own_event` says whether anything actually happened to this name.
2. **A name's own value moves with no new event.** The insider signal is a
   *trailing sum over a rolling window* ÷ trailing dollar volume — so an old buy
   rolling off the back of the window moves the score by itself. "Own-driven" and
   "an event happened" are different claims, and are reported separately.
3. **A missing signal is not a zero — it is a neutral 0.5.** `filled_*` marks
   bars where the name had *no data* for a sleeve and was handed the median rank.
   It looks scored; it carries no information. (This is not hypothetical: it is
   how a 3-of-100-symbol `short_interest` table silently became a tilt on three
   mega-caps.)

**Timing, honestly.** Features are lagged one bar, and fills are next-bar-open. So
an event known on day *D* lands on feature bar *D+1* and can fill no earlier than
the open of *D+2*. The figure draws the signal crossing as a **hollow circle** and
the actual fill as a **solid triangle** — the gap between them is that delay, made
literal. Shaded spans are **real positions read from `bt.trade_records`**, never a
reconstruction from the boolean frames: vectorbt ignores an entry while already
long, and an entry and exit landing on the same bar cancel each other, so signals
and positions genuinely can disagree. Where they do, this figure shows it. Note that
`trade_records` holds one row per FIFO **lot**, not per position — a name that is
scaled out of has many rows sharing a single entry — so the lots are unioned back
into the intervals the book was actually long before they are drawn.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from trading_simple.plots import DIVERGING, SERIES

# explain() is not on LoadedStrategy (the loader binds only signals/valid/indicators),
# so reach the module the engine already executed — no re-import, no sys.path games.
explain = strat.signals.__globals__["explain"]

PARAMS = strat.params          # swap for `picked` after §7 to explain the tuned run
PAD = 60                       # bars either side of the crossing we zoom on

trades = bt.trade_records(res)
# Explain the name the strategy traded most — that is where there is something to see.
SYMBOL = (trades["symbol"].value_counts().idxmax() if len(trades)
          else panel.features["insider"].abs().sum().idxmax())

exp = explain(panel, SYMBOL, **PARAMS)
mine = trades[trades["symbol"] == SYMBOL]

# Zoom on the first entry; fall back to the strongest rank jump if it never traded.
focus = (pd.Timestamp(mine["entry_ts"].iloc[0]) if len(mine)
         else exp["d_score"].abs().idxmax())
start = max(exp.index.get_indexer([focus], method="nearest")[0] - PAD, 0)
win = exp.iloc[start: start + 2 * PAD]

print(f"{SYMBOL}: {len(mine)} trades, zooming on {focus.date()} ±{PAD} bars")
print(f"sleeves in the composite: {[c[5:] for c in exp.columns if c.startswith('rank_')]}")
if (dropped := set(panel.features) - {c[5:] for c in exp.columns if c.startswith('rank_')}):
    print(f"⚠ dropped (zero weight, or below the coverage floor): {sorted(dropped)}")

In [ ]:
SLEEVES = [c[5:] for c in exp.columns if c.startswith("rank_")]
q_in, q_out = PARAMS["q_in"], PARAMS["q_out"]

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, row_heights=[0.40, 0.28, 0.32],
    vertical_spacing=0.06, specs=[[{}], [{}], [{"secondary_y": True}]],
    subplot_titles=("① Composite rank vs the q_in / q_out bands",
                    "② Why: each sleeve's weighted contribution (stacked = composite)",
                    "③ The events that moved it, as the strategy saw them"))
fig.update_annotations(font_size=13)   # the 3 subplot titles — the only annotations so far

# --- ① the rank the bands are compared against ----------------------------
fig.add_trace(go.Scatter(x=win.index, y=win["score_pct"], name="composite rank",
                         line={"color": SERIES[0], "width": 2},
                         customdata=np.c_[win["own"], win["peer"], win["driver"]],
                         hovertemplate=("rank %{y:.2f}<br>own %{customdata[0]:+.3f}"
                                        " · peer %{customdata[1]:+.3f}"
                                        " → %{customdata[2]}-driven<extra></extra>")),
              row=1, col=1)
# Qualified (hollow) vs actually filled (solid). The gap is TWO different things now:
# the D+2 execution delay, AND the capital constraint — a qualified name with no free
# slot is never bought at all. Those bars are marked separately, in red.
cross = win.index[win["qualified"]]
fig.add_trace(go.Scatter(
    x=cross, y=win["score_pct"].reindex(cross), mode="markers", name="qualified",
    marker={"symbol": "circle-open", "size": 13, "color": SERIES[3],
            "line": {"width": 2, "color": SERIES[3]}}), row=1, col=1)
# A "-thin" symbol has no fill — plotly strokes it from marker.line.color, so setting only
# marker.color leaves it the default slate: this ✕ was never actually red on screen.
if (turned_away := win.index[win["slot_blocked"]]).size:
    fig.add_trace(go.Scatter(
        x=turned_away, y=win["score_pct"].reindex(turned_away), mode="markers",
        name="NO SLOT (book full)",
        marker={"symbol": "x-thin", "size": 10, "color": "#c0392b",
                "line": {"width": 2.5, "color": "#c0392b"}}), row=1, col=1)
w0, w1 = win.index[0], win.index[-1]
# vectorbt records one row per LOT (FIFO), not per position: a name that is scaled out of
# has many rows sharing ONE entry with different exits. Dedupe to the bars the book
# actually opened on, or 25 lots stamp 25 triangles on the same day.
fills = pd.DatetimeIndex(sorted(set(pd.to_datetime(mine["entry_ts"])))).intersection(win.index)
# Indigo, not the green of the circles: the hollow-circle → solid-triangle gap IS the point
# of this panel, and a green triangle under a green circle of the same size cannot be seen.
fig.add_trace(go.Scatter(
    x=fills, y=win["score_pct"].reindex(fills), mode="markers", name="actual fill",
    marker={"symbol": "triangle-up", "size": 15, "color": SERIES[4],
            "line": {"width": 1.5, "color": "#ffffff"}}), row=1, col=1)
# Ineligible bars: a top rank here still cannot trade (min_events).
if (blocked := win.index[~win["eligible"]]).size:
    fig.add_trace(go.Scatter(
        x=blocked, y=np.zeros(len(blocked)), mode="markers", name="ineligible (min_events)",
        marker={"symbol": "line-ns-open", "size": 8, "color": "#898781",
                "line": {"width": 1.5, "color": "#898781"}}), row=1, col=1)

# --- ② attribution; hatched = neutral-filled (NO DATA) ---------------------
for i, k in enumerate(SLEEVES):
    gap = win[f"filled_{k}"]
    fig.add_trace(go.Bar(
        x=win.index, y=win[f"contrib_{k}"], name=k, marker_color=SERIES[i],
        marker_pattern={"shape": ["/" if g else "" for g in gap],
                        "fgcolor": "#ffffff", "solidity": 0.35},
        customdata=np.c_[win[f"rank_{k}"], gap],
        hovertemplate=(f"<b>{k}</b><br>contribution %{{y:.3f}}<br>rank %{{customdata[0]:.2f}}"
                       "<br>no data (neutral 0.5): %{customdata[1]}<extra></extra>")),
        row=2, col=1)
fig.add_trace(go.Scatter(x=win.index, y=win["composite"], name="composite",
                         line={"color": "#0b0b0b", "width": 1.5}), row=2, col=1)

# --- ③ the raw point-in-time events ---------------------------------------
if "insider_usd" in win:
    v = win["insider_usd"].fillna(0)
    fig.add_trace(go.Bar(x=win.index, y=v, name="insider net $",
                         marker_color=[DIVERGING[2][1] if x > 0 else DIVERGING[0][1] for x in v],
                         hovertemplate="insider $%{y:,.0f}<extra></extra>"), row=3, col=1)
for col, label, shape in [("rating_change", "rating Δ", "triangle-up"),
                          ("ptrev", "price-target rev", "diamond")]:
    if col not in win:
        continue
    ev = win[col].fillna(0)
    ev = ev[ev != 0]
    fig.add_trace(go.Scatter(
        x=ev.index, y=np.zeros(len(ev)), mode="markers", name=label,
        marker={"symbol": [shape if x > 0 else shape.replace("-up", "-down") for x in ev],
                "size": 10, "color": [SERIES[3] if x > 0 else SERIES[5] for x in ev]},
        customdata=ev.to_numpy(),
        hovertemplate=f"{label} %{{customdata:+.3f}}<extra></extra>"), row=3, col=1)
if "days_to_cover" in win:
    fig.add_trace(go.Scatter(x=win.index, y=win["days_to_cover"], name="days-to-cover",
                             line={"color": SERIES[6], "width": 1.5, "shape": "hv"}),
                  row=3, col=1, secondary_y=True)

# --- the bands and the real positions, drawn UNDER everything above --------
# These must be added AFTER the traces: add_hrect/add_vrect default to
# exclude_empty_subplots=True, so a shape aimed at a still-traceless subplot is
# dropped without warning. `layer="below"` keeps them behind regardless of order.
fig.add_hrect(y0=q_in, y1=1.0, fillcolor=SERIES[1], opacity=0.10, line_width=0,
              layer="below", row=1, col=1,
              annotation_text=f"IN (rank ≥ q_in = {q_in})", annotation_position="top left",
              annotation_font_size=11, annotation_font_color="#898781")
fig.add_hrect(y0=q_out, y1=q_in, fillcolor=SERIES[2], opacity=0.09, line_width=0,
              layer="below", row=1, col=1,
              annotation_text=f"HOLD band (q_out = {q_out} – {q_in})",
              annotation_position="bottom left",
              annotation_font_size=11, annotation_font_color="#898781")
# REAL positions, from the portfolio — not reconstructed. Union the FIFO lots back into
# the intervals the book was actually long: 34 overlapping lot-rects would stack their
# 0.08 fills into a solid block, and their exits run to 2024, dragging the axis right off
# the ±PAD zoom. Clip what survives to the window.
lots = mine[["entry_ts", "exit_ts"]].apply(pd.to_datetime).sort_values("entry_ts")
held: list[list[pd.Timestamp]] = []
for entry, exit_ in lots.itertuples(index=False):
    if held and entry <= held[-1][1]:                 # overlaps the open span → extend it
        held[-1][1] = max(held[-1][1], exit_)
    else:
        held.append([entry, exit_])
for entry, exit_ in held:
    if exit_ >= w0 and entry <= w1:
        fig.add_vrect(x0=max(entry, w0), x1=min(exit_, w1),
                      fillcolor=SERIES[3], opacity=0.08, line_width=0, layer="below")

fig.update_layout(
    title={"text": (f"<b>{SYMBOL}</b> — why it is in or out of the book<br>"
                    "<sup>○ qualified · ✕ no slot · ▲ actual fill · shaded span = position held"
                    " · hatched bar = NO DATA, neutral 0.5</sup>"),
           "font": {"size": 15, "color": "#0b0b0b"},
           "x": 0, "xref": "paper", "xanchor": "left",
           "y": 0.97, "yref": "container", "yanchor": "top"},
    template="plotly_white", height=940, barmode="relative", bargap=0.05,
    font={"family": 'system-ui, -apple-system, "Segoe UI", sans-serif', "size": 12},
    hovermode="x unified",
    # its own band above the panels: a 14-entry horizontal legend wraps to two rows,
    # and anchored at the top right it lands on top of subplot title ①.
    legend={"orientation": "h", "x": 0.5, "xanchor": "center",
            "y": 1.06, "yanchor": "bottom", "font": {"size": 11}},
    margin={"l": 70, "r": 70, "t": 170, "b": 50})
fig.update_xaxes(range=[w0, w1])       # pin the zoom: a shape must not drag the autorange
fig.update_yaxes(title_text="rank (0–1)", range=[-0.02, 1.02], row=1, col=1)
fig.update_yaxes(title_text="contribution", row=2, col=1)
fig.update_yaxes(title_text="insider $ / events", row=3, col=1)
fig.update_yaxes(title_text="days-to-cover", showgrid=False, row=3, col=1, secondary_y=True)
fig.show()

In [ ]:
# Every bar this name crossed a band, with the cause attributed. `driver == "peer"`
# together with `own_event == False` is the counter-intuitive case: the name crossed
# because the CROSS-SECTION moved under it — nothing happened to the company.
zone_prev = exp["zone"].shift(1)
crossings = exp[(exp["zone"] != zone_prev) & zone_prev.notna()].copy()
crossings["move"] = zone_prev[crossings.index] + " → " + crossings["zone"]

cols = ["move", "score_pct", "d_score", "own", "peer", "driver", "own_event", "eligible"]
cols += [f"d_contrib_{k}" for k in SLEEVES]
display(crossings[cols].round(3).tail(12))

peer_driven = crossings[(crossings["driver"] == "peer") & ~crossings["own_event"]]
print(f"{len(crossings)} band crossings for {SYMBOL}; "
      f"{len(peer_driven)} were pure PEER DRIFT — the name crossed with no news of its own.")
if len(peer_driven):
    # NB: not `d` — that name holds the strategy's data config for the whole notebook.
    last = peer_driven.index[-1].date()
    print(f"  most recent: {last} — nothing happened to {SYMBOL}; its peers moved.")

## 4b · The whole book at once — who is held, and when

The per-name view answers *why this one*. This answers *what is the portfolio doing* —
every symbol's daily state in one frame:

- **held** — an actual open position (from `bt.trade_records`, so this is truth).
- **slot_blocked** — ranked above `q_in` and **not bought, because the book was full**.
  This state did not exist under per-symbol cash accounts, where every name had its own
  money and a qualifying name was *always* bought. It is the visible face of the capital
  constraint, and its size is the answer to "its rank is 0.95, why do I not own it?" —
  the money is already spent. If names are blocked constantly, `max_positions` is the
  binding parameter, not `q_in`.
- **band** — inside the `q_out`–`q_in` hysteresis zone: held names stay held here, flat
  names stay flat. This is the turnover damper — and it is also the *only* place
  `max_hold` can bite, since a name back above `q_in` has a fresh thesis and its clock
  resets.
- **out** — below `q_out`.
- **ineligible** — fewer than `min_events` insider/rating events in the trailing window.
  These names are **not ranked at all**: they are removed from the cross-section, so they
  neither get a score nor displace anyone else's percentile.

Read it for **churn** (names flickering in and out are paying costs for nothing), for
**breadth** (if the same handful of names is always held, the cross-sectional thesis is
not doing any work), and for **pressure** (a wall of `slot_blocked` means the ranking has
more conviction than the book has room for).


In [ ]:
# Rebuild the whole book at once, through the SAME code path the backtest ran
# (_compose -> _select), so this picture cannot drift from what was actually traded.
mod = strat.signals.__globals__
compose = mod["_compose"]        # also used by §4c below
parts = compose(panel, **PARAMS)
selected = mod["_select"](parts, q_in=PARAMS["q_in"], q_out=PARAMS["q_out"],
                          max_hold=PARAMS["max_hold"],
                          max_positions=PARAMS["max_positions"],
                          displace=PARAMS["displace"])
score, eligible = parts["score_pct"], parts["eligible"]

# 0 ineligible · 1 out · 2 band · 3 qualified but NO SLOT · 4 held
# score is NaN for ineligible names (dropped from the cross-section), and a NaN
# comparison is False, so they fall through to 1 and are then overwritten with 0.
state = np.where(score >= q_in, 3, np.where(score >= q_out, 2, 1))
state = np.where(eligible.to_numpy(), state, 0)
state = pd.DataFrame(state, index=score.index, columns=score.columns)
state = state.mask(selected["slot_blocked"], 3)
for _, t in trades.iterrows():   # truth beats reconstruction
    state.loc[pd.Timestamp(t["entry_ts"]):pd.Timestamp(t["exit_ts"]), t["symbol"]] = 4

order = state.eq(4).sum().sort_values(ascending=False).index   # most-held names first
z = state[order].T
LABELS = ["ineligible", "out", "band", "qualified — NO SLOT", "held"]
COLORS = ["#e8e6df", "#f4f3ef", "#fde3c0", "#e8a49c", SERIES[3]]
bands = [[i / 5, COLORS[j]] for j in range(5) for i in (j, j + 1)]

fig = go.Figure(go.Heatmap(
    z=z.to_numpy(), x=z.columns, y=z.index, colorscale=bands, zmin=0, zmax=5,
    colorbar={"tickvals": [0.5, 1.5, 2.5, 3.5, 4.5], "ticktext": LABELS, "len": 0.6},
    hovertemplate="%{y} · %{x|%Y-%m-%d}<extra></extra>"))
fig.update_layout(
    title=(f"Book state by name — {len(order)} symbols · q_in={q_in}, q_out={q_out} · "
           f"{PARAMS['max_positions']} slots · held spans are real trades"),
    template="plotly_white", height=max(460, 11 * len(order)),
    margin={"l": 70, "r": 20, "t": 70, "b": 40})
fig.update_yaxes(tickfont={"size": 8})
fig.show()

held_days = int(state.eq(4).sum().sum())
blocked_days = int(state.eq(3).sum().sum())
print(f"held: {held_days:,} symbol-days across {int(state.eq(4).any().sum())} names")
print(f"qualified but NO SLOT: {blocked_days:,} symbol-days — "
      f"{selected['slot_blocked'].sum(axis=1).mean():.1f} names turned away on an average "
      f"day. The book has {PARAMS['max_positions']} slots and the ranking wants more; "
      "that gap is the capital constraint doing its job.")
print(f"names ranked per day: {parts['n_ranked'].mean():.0f} of {len(score.columns)} "
      "(the rest fall below min_events and are dropped from the cross-section entirely)")


### 4b′ · Do the gates thin the flood of qualifiers?

This is the whole motivation. Re-run the **same** `_compose` result through `_select`
with the new gates off (= `informed_flow`) versus a sample gated set, and compare how
many names are turned away for want of a slot on an average day. A good gate pulls the
qualifying set toward the number of slots, so the book chooses among genuine contenders
instead of rationing a flood. (This is a *diagnostic*, not the tuned pick — §7 searches
which gates actually pay.)


In [ ]:
def gate_pressure(**gate):
    """Avg names blocked/day and distinct names ever held, for a gate setting."""
    sel = mod["_select"](parts, q_in=PARAMS["q_in"], q_out=PARAMS["q_out"],
                         max_hold=PARAMS["max_hold"], max_positions=PARAMS["max_positions"],
                         displace=PARAMS["displace"], **gate)
    return float(sel["slot_blocked"].sum(axis=1).mean()), int(sel["held"].any().sum())

b0, n0 = gate_pressure()                                   # gates off == informed_flow
b1, n1 = gate_pressure(min_quality_pct=0.5, min_agree=2)   # top-half quality + 2 agree
print(f"gates OFF (= informed_flow): {b0:5.1f} blocked/day · {n0} names ever held")
print(f"gates ON  (qual>=0.5, ag=2): {b1:5.1f} blocked/day · {n1} names ever held")
print(f"\nbook has {PARAMS['max_positions']} slots. Fewer blocked/day means the ranking's")
print("conviction is closer to the book's room — choosing contenders, not rationing a flood.")
if parts.get("quality_pct") is None:
    print("\n\u26a0 fundamentals below the coverage floor — the quality gate self-disabled.")


## 4c · Does the ranking predict anything at all?

**Run this before trusting any backtest number.** It asks the only question that
matters for a cross-sectional strategy — *does a higher rank mean a higher future
return?* — and it answers it without the trading engine, so the answer cannot be
contaminated by exposure, costs, or exit rules.

The trick is to **subtract each day's cross-sectional mean from the forward
returns**. What is left is pure stock selection: if the whole market rises, that
rise is removed. This matters enormously, because a long-only strategy in a rising
market shows a healthy Sharpe ratio whether or not its stock-picking works at all.
A backtest cannot separate the two. This can.

Two measures, both on the **training window only**:

- **Rank IC** — the correlation between today's rank and the future relative
  return. Anything from +0.03 upward is respectable for this kind of signal; zero
  means the ranking is noise.
- **Decile spread** — what the top 10% of names earned, relative to the market,
  minus what the bottom 10% earned. This is the money question, and it should be
  positive and should *grow* with the horizon if the signal is real.

Each sleeve is scored separately, so you can see which ones are carrying the
composite and which are dead weight. A sleeve with a **negative** spread is not
neutral — it is actively pulling the composite the wrong way.

In [ ]:
HORIZONS = (21, 63, 126)          # ~1, 3 and 6 months ahead

def signal_quality(score: pd.DataFrame, close: pd.DataFrame) -> pd.DataFrame:
    """Rank IC and decile spread of a score against DEMEANED forward returns.

    Demeaning each day's forward returns cross-sectionally removes the market, so what
    remains measures stock selection alone — which is exactly what a backtest of a
    long-only strategy cannot show you.

    The t-statistic is divided by sqrt(horizon): consecutive daily ICs share almost all
    of their forward window (today's 63-day return overlaps tomorrow's by 62 days), so
    they are nowhere near independent observations. The naive t-stat is inflated by
    roughly sqrt(h) and would report overwhelming significance for a signal that has
    none. This correction is crude but honest; the naive number is a lie.
    """
    rows = []
    r = score.rank(axis=1, pct=True)
    for h in HORIZONS:
        fwd = close.shift(-h) / close - 1.0
        rel = fwd.sub(fwd.mean(axis=1), axis=0)          # <- the market, removed
        ic = score.corrwith(rel, axis=1, method="spearman").dropna()
        naive_t = ic.mean() / (ic.std() / np.sqrt(len(ic))) if len(ic) and ic.std() else 0.0
        rows.append({
            "horizon": f"{h}d",
            "rank_IC": round(float(ic.mean()), 4),
            "t_stat (overlap-adj)": round(float(naive_t / np.sqrt(h)), 2),
            "top_decile": float(rel.where(r >= 0.9).mean(axis=1).mean()),
            "bottom_decile": float(rel.where(r <= 0.1).mean(axis=1).mean()),
        })
    out = pd.DataFrame(rows).set_index("horizon")
    out["spread"] = out["top_decile"] - out["bottom_decile"]
    return out

FMT = {"top_decile": "{:+.2%}", "bottom_decile": "{:+.2%}", "spread": "{:+.2%}"}
parts_q = compose(panel, **PARAMS)
print("COMPOSITE — the score the strategy actually trades")
display(signal_quality(parts_q["score_pct"], panel.close).style.format(FMT))

print("\nPER SLEEVE — who is carrying the composite, and who is dead weight?")
summary = {}
for k in parts_q["present"]:
    q = signal_quality(parts_q["ranks"][k], panel.close)
    summary[k] = {"63d IC": q.loc["63d", "rank_IC"], "63d spread": q.loc["63d", "spread"],
                  "126d IC": q.loc["126d", "rank_IC"], "126d spread": q.loc["126d", "spread"]}
display(pd.DataFrame(summary).T.style.format(
    {"63d spread": "{:+.2%}", "126d spread": "{:+.2%}",
     "63d IC": "{:+.4f}", "126d IC": "{:+.4f}"}))
print("A NEGATIVE spread is not neutral — that sleeve is pulling the composite the wrong")
print("way, and a weight above zero on it is actively costing you money.")
print("\nCAUTION: a positive decile spread here does NOT mean the strategy can harvest it.")
print("The spread is measured on the TOP 10% at a FIXED horizon. If the strategy holds the")
print("top 40%, or holds a name long after its rank has decayed, the edge is diluted away —")
print("which is exactly what happened. Section 10a is the test that matters.")

## 5 · Coarse sweep

A small grid to see the shape of the objective before the optimizer runs. The
`w_dtc: [0.0, 0.3]` column ablates short interest, so its marginal value is
visible side by side.

In [ ]:
grid = bt.grid_from_config(strat.sweep, strat.valid)
sweep = bt.sweep(strat, panel, grid, settings=settings)
display(sweep.sort_values("sharpe", ascending=False).head(10).round(3))
plots.plot_sweep(sweep).show()

## 6 · Optimize (Optuna, resumable)

Maximizes Sharpe **and** total return together (a Pareto front). The study is
journal-backed, so this cell resumes if re-run. `N_OPTIMIZE_TRIALS` is a knob —
raise it for a finer front, but remember every extra trial is another chance to
overfit (the deflated Sharpe in §10 accounts for exactly this).

In [ ]:
N_OPTIMIZE_TRIALS = 1000   # five extra levers => a larger space; run the sweep first
jobs.run_optimize("conviction_flow", n_trials=N_OPTIMIZE_TRIALS)


## 7 · Read the study — admissibility before picking

Few-trade parameter sets can post a flattering Sharpe by luck, so we filter the
Pareto front to trials with at least `MIN_TRADES` before choosing (the
convention from `squeeze_breakout`). Among the admissible front we take the
Sharpe-tilted weighted best.

In [ ]:
MIN_TRADES = 100
study = opt.load_study(strat)
front = opt.pareto_frame(study)
admissible = front[front["n_trades"] >= MIN_TRADES].copy()
print(f"front: {len(front)} trials, admissible (n_trades ≥ {MIN_TRADES}): {len(admissible)}")

_INTS = ("insider_window", "rating_halflife", "ptrev_halflife", "max_hold", "min_events",
         "max_positions")
def coerce(params):
    """Cast params back to their real types (front rows store everything as object/float)."""
    out = {}
    for k, v in params.items():
        out[k] = int(round(v)) if k in _INTS else bool(v) if k == "displace" else float(v)
    return out

# Pick on the study's own objectives — both are measured against buy-and-hold, so a
# trial only scores above zero by beating the market. Picking on absolute sharpe here
# would quietly undo the whole point of the benchmark-relative objective. One rule for the
# whole repo (opt.PICK_WEIGHTS, balanced), and it collapses tied outcomes first.
X, Y = opt.OBJECTIVES                       # (info_ratio, excess_cagr)
pool = admissible if len(admissible) else front
best_row = opt.pick_from_front(pool)
print(f"{int(best_row['front_distinct'])} distinct outcomes on the admissible front"
      + (f" — picked one of {int(best_row['n_tied'])} TIED trials, arbitrarily"
         if best_row["n_tied"] > 1 else ""))
picked = coerce({k: best_row[k] for k in strat.optuna["space"]})

beats = (front[X] > 0) & (front[Y] > 0)
print(f"trials on the front that BEAT buy-and-hold: {int(beats.sum())} of {len(front)}")
print("picked params:", picked)
# `years` and `trades_per_year` are not decoration: excess_cagr is ANNUALIZED, which
# makes a short window comparable but never trustworthy. Always read the sample size.
print("picked train metrics:", {m: round(float(best_row[m]), 4)
      for m in (X, Y, "sharpe", "cagr", "pct_profitable", "n_trades",
                "trades_per_year", "years")})
plots.plot_pareto(front, opt.trials_frame(study), picked=int(best_row["trial"])).show()

## 8 · Boundary check

If a picked value sits on a search-space edge, the real optimum may lie outside
the box — widen that bound and re-run. (Edge flags driven only by the thin
few-trade tail are false positives; judge against the admissible front.)

In [ ]:
report = opt.boundary_report(picked, strat.optuna["space"], front=front)
display(report)

## 9 · Backtest the picked parameters

In [ ]:
res_best = bt.run_strategy(strat, panel, params=picked, settings=settings)
stats_best = bt.portfolio_stats(res_best)
for k, v in stats_best.items():
    print(f"  {k:16s} = {v:.4f}")
plots.plot_equity(bt.equity_curve(res_best)).show()
display(bt.symbol_stats(res_best).sort_values("total_return", ascending=False).head(10).round(3))

## 10 · Validate — is the edge real?

Everything above is in-sample. This section is the honest test, and **§10a is the
one that can kill the strategy outright**, so it comes first. Knobs below (lowered
here run faster; raise for a stricter check).

In [ ]:
N_RANDOM_SIMS = 200            # random-entry baseline simulations
COST_MULTIPLES = [1.0, 2.0, 3.0]  # cost-stress multiples

def slice_panel(p, start, end):
    """A copy of the panel restricted to [start, end], features sliced too."""
    m = ((p.close.index >= pd.Timestamp(start, tz="UTC"))
         & (p.close.index <= pd.Timestamp(end, tz="UTC")))
    idx = p.close.index[m]
    feats = {k: v.loc[idx] for k, v in (p.features or {}).items()}
    return replace(p, close=p.close.loc[idx], open_=p.open_.loc[idx], high=p.high.loc[idx],
                   low=p.low.loc[idx], volume=p.volume.loc[idx], features=feats)

### 10a · The hurdle: does it beat simply owning the universe?

The first question, and the one most easily skipped. A long-only strategy in a
rising market earns a positive Sharpe ratio **merely by being invested**. So an
absolute Sharpe of 0.9 sounds excellent and can still be a failure, because holding
every stock in the universe might have returned more, with less effort and fewer
costs.

The reference is an **equal-weight buy-and-hold** of the same names over the same
period, paying the same costs. If the strategy does not beat it, the stock selection
is adding nothing and there is no reason to run it — regardless of how healthy the
headline numbers look.

This is also what the optimizer now maximizes (`info_ratio` and `excess_cagr`,
both measured against this benchmark), rather than absolute Sharpe. The earlier
version of this strategy maximized absolute Sharpe, and the optimizer duly learned
to make the strategy resemble the index: it drove the entry quantile to the least
selective value it was allowed and still lost to buy-and-hold on both the training
data and the holdout.

In [ ]:
def hurdle(p, params, label):
    """Strategy vs equal-weight buy-and-hold of the same panel. The pass/fail gate.

    Both sides are now the same construction: one shared $10k book, fully invested,
    paying the same costs. Under per-symbol cash accounts they were not — the strategy
    was ~30% invested against a 100%-invested benchmark, and lost on that alone.
    """
    st = bt.evaluate(strat, p, params, settings=settings)
    fees, slip = bt.panel_costs(settings.costs, p)
    bench_ret, _ = bt.benchmark(p, fees=fees, slippage=slip)
    bench_cagr = bt._cagr(bench_ret, st["years"])
    ok = st["excess_cagr"] > 0 and st["info_ratio"] > 0
    verdict = ("PASS — it beats owning the universe" if ok
               else "FAIL — owning the universe was better")
    print(f"\n{label}  ({p.close.index[0].date()} -> {p.close.index[-1].date()}, "
          f"{st['years']:.2f} years, {st['trades_per_year']:.0f} trades/yr)")
    print(f"  strategy     {st['cagr']:+7.2%}/yr   sharpe {st['sharpe']:5.2f}   "
          f"n_trades {int(st['n_trades'])}")
    print(f"  buy & hold   {bench_cagr:+7.2%}/yr")
    print(f"  EXCESS       {st['excess_cagr']:+7.2%}/yr   "
          f"information ratio {st['info_ratio']:+5.2f}")
    print(f"  --> {verdict}")
    return st

hurdle(panel, picked, "TRAIN")
full_h = load_panel(engine, syms, d["bar"], d["start"], None,
                    session=d.get("session", "regular"), exchanges=settings.exchanges)
features.attach_features(full_h, engine, d["features"], settings=settings)
holdout = slice_panel(full_h, "2025-01-01", str(full_h.close.index[-1].date()))
hurdle(holdout, picked, "HOLDOUT")

# A holdout number is only as good as the window under it. Annualizing a part-year stub
# turns a good six months into a spectacular "per year" figure that means very little, so
# slice the holdout and look at each complete year on its own before believing the total.
for a, b, label in [("2025-01-01", "2025-12-31", "2025 (complete year)"),
                    ("2026-01-01", str(full_h.close.index[-1].date()), "2026 (part year)")]:
    sub = slice_panel(full_h, a, b)
    if len(sub.close) < 40:
        continue
    hurdle(sub, picked, f"  HOLDOUT · {label}")


### 10a · Costs on, then doubled — and the breakeven

Never trust a zero-cost result. Re-run the pick at 1×/2×/3× the configured
costs, then find the round-trip cost (in bps) at which the Sharpe hits zero. An
edge that dies at 2× costs was trading noise around the cost hurdle.

In [ ]:
def with_costs(**overrides):
    """A copy of settings with its (pydantic) cost block overridden."""
    return settings.model_copy(update={"costs": settings.costs.model_copy(update=overrides)})

c = settings.costs
for mult in COST_MULTIPLES:
    s = with_costs(spread_bps=c.spread_bps * mult, slippage_bps=c.slippage_bps * mult,
                   commission_per_share=c.commission_per_share * mult)
    st = bt.portfolio_stats(bt.run_strategy(strat, panel, params=picked, settings=s))
    print(f"  {mult:.0f}x costs: sharpe={st['sharpe']:.3f}  return={st['total_return']:.2%}")

base_bps = c.spread_bps + c.slippage_bps
breakeven = None
for extra in np.arange(0, 60, 2.0):
    s = with_costs(spread_bps=base_bps + extra, slippage_bps=0.0)
    st = bt.portfolio_stats(bt.run_strategy(strat, panel, params=picked, settings=s))
    if st["sharpe"] <= 0:
        breakeven = base_bps + extra
        break
print(f"\napprox breakeven half-spread: "
      f"{'>60' if breakeven is None else round(breakeven,1)} bps per leg")

### 10b · Walk-forward

Expanding-window: re-pick the best sweep config on each training slice, then
score it on the *next* unseen year. The concatenated out-of-sample Sharpes are
the number that matters — in-sample Sharpe is free.

In [ ]:
# Training starts 2021 (config data.start), leaving 4 folds. Each fold re-picks on
# EXCESS RETURN (over buy-and-hold), not raw Sharpe — picking on the absolute number
# would select the most market-exposed combo in every fold and tell us nothing.
FOLDS = [("2021-01-01", "2022-12-31", "2023-01-01", "2023-06-30"),
         ("2021-01-01", "2023-06-30", "2023-07-01", "2023-12-31"),
         ("2021-01-01", "2023-12-31", "2024-01-01", "2024-06-30"),
         ("2021-01-01", "2024-06-30", "2024-07-01", "2024-12-31")]
is_x, oos_x = [], []
for tr0, tr1, te0, te1 in FOLDS:
    tr = bt.sweep(strat, slice_panel(panel, tr0, tr1), grid, settings=settings)
    tr = tr[tr["n_trades"] >= 30]
    row = tr.sort_values("excess_cagr", ascending=False).iloc[0]
    pk = coerce({k: row[k] for k in grid[0]})
    st = bt.evaluate(strat, slice_panel(panel, te0, te1), pk, settings=settings)
    is_x.append(float(row["excess_cagr"]))
    oos_x.append(float(st["excess_cagr"]))
    print(f"  test {te0[:7]}..{te1[:7]}: IS excess {row['excess_cagr']:+.2%} "
          f"→ OOS excess {st['excess_cagr']:+.2%}  (OOS sharpe {st['sharpe']:.2f})")
t, p = stats.ttest_rel(is_x, oos_x)
print(f"\nmean IS excess {np.mean(is_x):+.2%}  vs  mean OOS excess {np.mean(oos_x):+.2%}")
print(f"  (paired t={t:.2f}, p={p:.2f} over {len(FOLDS)} folds — few folds, so a high p is")
print("   weak evidence, not a clean bill of health)")
print(f"  folds where the strategy BEAT buy-and-hold out of sample: "
      f"{sum(x > 0 for x in oos_x)}/{len(oos_x)}")

### 10c · Deflated Sharpe Ratio

The best of N optimizer trials beats a naive threshold by luck alone. The
Deflated Sharpe (Bailey & López de Prado 2014) discounts the picked Sharpe for
the number of trials tried and the return distribution's skew/kurtosis. Want
DSR > 0.95; because TPE/NSGA trials are correlated, we report it across a band
of effective trial counts.

In [ ]:
# Deflate the metric we SELECTED ON (info_ratio), using the ACTIVE returns it is built
# from — deflating absolute Sharpe would be answering a question we did not ask.
tf = opt.trials_frame(study)
trial_ir = tf["info_ratio"].dropna().to_numpy()
fees_, slip_ = bt.panel_costs(settings.costs, panel)
_, bench_curve = bt.benchmark(panel, fees=fees_, slippage=slip_)
active = (bt.equity_curve(res_best)["value"].pct_change().fillna(0.0) - bench_curve).to_numpy()
ann = np.sqrt(252)

def deflated(trial_scores, score_ann, returns, n_eff):
    """Bailey & Lopez de Prado (2014): P(the pick is genuinely > 0 given N tries)."""
    gamma = 0.5772156649
    sd = np.std(trial_scores, ddof=1)
    e_max = sd * ((1 - gamma) * stats.norm.ppf(1 - 1.0 / n_eff)
                  + gamma * stats.norm.ppf(1 - 1.0 / (n_eff * np.e)))
    sr, sr0 = score_ann / ann, e_max / ann
    sk, ku = stats.skew(returns), stats.kurtosis(returns, fisher=False)
    denom = np.sqrt(1 - sk * sr + (ku - 1) / 4 * sr ** 2)
    return float(stats.norm.cdf((sr - sr0) * np.sqrt(len(returns) - 1) / denom))

ir = float(best_row["info_ratio"])
n = len(trial_ir)
print(f"trials scored: {n};  picked INFORMATION RATIO (vs buy-and-hold): {ir:.3f}")
for n_eff in sorted({max(2, n // 4), max(2, n // 2), n}):
    print(f"  deflated at n_eff={n_eff:4d}: {deflated(trial_ir, ir, active, n_eff):.3f}")
print("\nWant > 0.95. This asks: after trying this many parameter sets, how confident are")
print("we that the winner's edge OVER BUY-AND-HOLD is real rather than the luckiest draw?")

### 10d · Parameter-neighborhood stability — plateau or spike?

§10c asked whether the winner beat the *other trials*. This asks what the study
cannot: are the winning parameter **values** load-bearing? Perturb one knob at a
time by ±5% and ±10% of its own searched range — log-aware, snapped to the grid the
study could itself have suggested — and re-score. Train window only: re-centering a
pick is a training decision, and scoring candidates on the holdout would burn it.

**The two panels.**

- **Tornado (top)** — one bar per knob, spanning the worst and best its own
  perturbations produced. Long bar = the result hangs on that value; short bar =
  the knob is not doing work. A **red** bar reaches zero: some value one small step
  away hands the whole edge back to buy-and-hold.
- **Profiles (below)** — the metric against each knob's actual values, ◆ = the
  picked value. Flat top = plateau. A lone peak between two drops = the overfit
  spike. The pick sitting on the shoulder of a drop = a cliff to step away from.
  Each panel is zoomed to **its own** neighborhood, so read the y-axis before the
  shape: a knob worth 0.1pp and a knob worth 10pp both fill their panel, and the
  zero line only appears where the neighborhood actually reaches it. The tornado
  above is the comparable view.

**What to do about it** — one row per knob, and the `action` column says it:

| verdict | what it means | the action |
|---|---|---|
| `inert` | the whole neighborhood scores the same | pin the value and spend the trial budget on knobs that move the result. Mind the scope: this is a **local** claim — a knob flat within ±10% of its span can still be decisive at the far end of it, which is what §5's grid and the front are for |
| `plateau` | moves cost something, nothing breaks | ship it |
| `cliff (low/high)` | one side loses half the edge — or all of it — in one step | ship `suggest` instead: same plateau, further from the edge. An empty `suggest` means the pick is on a searched **bound** with the collapse right beside it — no room to step back, so the move is a wider bound (§8), not a re-centering |
| `spike` | *both* sides collapse | do not ship this pick; take a flatter point off the front |
| `off-center` | a neighbor beats the pick in sample | **not** an instruction to move — an in-sample bump is exactly what §10b exists to referee |
| `decisive` | (a categorical) the other choice loses the edge | keep it, and stop searching it |

`n_thin` counts neighbors falling below `MIN_TRADES`. A knob that starves the book
one step away is a finding about that knob, not a row to hide.

The metric is `excess_cagr`: benchmark-relative and per-year, one of the study's own
objectives, so 0 is the meaningful line rather than an arbitrary threshold. (The
other objective, `info_ratio`, needs a per-combo equity curve the column-stacked
sweep does not keep.)

> This replaced a cell that re-ran §5's *global* ablation grid and called it a
> neighborhood. It never evaluated a single point near `picked`, its heatmap
> marginalized over a **pinned** parameter (one row, nothing to see), and its
> plateau rule (`median > 0.5 × max`) inverts whenever the best neighbor is
> negative — half of a negative number is bigger than it, so a failing strategy
> reported a healthy plateau.

In [ ]:
# One coordinate at a time, ±5% and ±10% of each knob's own SEARCHED span. The star is
# one bt.sweep call, so the whole neighborhood costs about what the coarse grid does.
star = opt.sensitivity(strat, panel, picked, settings=settings)
report = opt.stability_report(star, min_trades=MIN_TRADES)

center = float(star.loc[star["perturbed"] == "", "excess_cagr"].iloc[0])
near = star.loc[star["perturbed"] != "", "excess_cagr"]
print(f"picked: {center:+.2%}/yr excess vs buy-and-hold; {len(near)} neighbors tried")
if center > 0:
    print(f"  still beating the benchmark: {int((near > 0).sum())}/{len(near)}"
          f"   ·   median neighbor keeps {near.median() / center:.0%} of the pick")
else:
    print("  the pick does not beat buy-and-hold — §10a already decided this; the shape of")
    print("  its neighborhood is not the problem.")
display(report)
plots.plot_sensitivity(star).show()

# The action, made concrete: every `cliff` knob stepped one value back onto its plateau.
# Scored on TRAIN, because that is the only window this decision may look at — take it to
# §10b's folds before pasting it into config.yaml `params:`.
# type(picked[k])(v): the star travels through a DataFrame, which upcasts an int knob to
# float as soon as any row differs — and signals() gets whatever it is handed.
moves = {k: type(picked[k])(v) for k, v in report["suggest"].dropna().items()}
if moves:
    recentred = {**picked, **moves}
    was, now = (bt.evaluate(strat, panel, p, settings=settings) for p in (picked, recentred))
    print("\nre-centered candidate — off the cliff edge, same plateau:")
    for k, v in moves.items():
        print(f"  {k}: {picked[k]} -> {v}")
    print(f"  excess_cagr {was['excess_cagr']:+.2%} -> {now['excess_cagr']:+.2%}/yr   "
          f"info_ratio {was['info_ratio']:+.2f} -> {now['info_ratio']:+.2f}   "
          f"trades/yr {was['trades_per_year']:.0f} -> {now['trades_per_year']:.0f}")
elif report["verdict"].str.startswith(("cliff", "spike")).any():
    print("\nnothing to re-center: the collapse sits against a searched bound, so there is")
    print("no safe step to take — read the action column, then §8.")
else:
    print("\nno knob sits on a cliff — the picked values are the ones to ship")

### 10e · Sub-period consistency

One number over five years can hide a strategy that only worked in one regime.
Score the pick on each calendar year and read the table, not a pass/fail.

In [ ]:
# Training starts 2021 (config data.start). If you widen it back to 2020 — the data is
# already ingested, so it is a one-line config change — add that year here: this table
# is exactly how you check whether the older regime helps or hurts, rather than guessing.
years = []
for y in range(2021, 2025):
    sub = slice_panel(panel, f"{y}-01-01", f"{y}-12-31")
    st = bt.evaluate(strat, sub, picked, settings=settings)
    years.append({"year": y, **{m: round(float(st[m]), 3)
                  for m in ("total_return", "sharpe", "pct_profitable", "n_trades")}})
display(pd.DataFrame(years).set_index("year"))

### 10f · Random-entry baseline

If random entries with the same exit logic and trade count do about as well, the
signal adds nothing. The pick should beat the 95th percentile of the random
distribution.

In [ ]:
# Random baseline, rebuilt for portfolio mode — and it is a far sharper test than the
# old one. Same book, same slots, same hysteresis, same turnover: only the RANKING is
# replaced by noise. If a random book scores like the real one, the edge was coming from
# the construction (concentration, staying invested) and the signal deserves no credit.
mod = strat.signals.__globals__
fees, slippage = bt.panel_costs(settings.costs, panel)
# `picked` holds only the SEARCHED params; w_dtc is the fixed numeraire and
# lives in config `params`. Merge, or _compose is missing an argument.
full_params = {**strat.params, **picked}
parts = mod["_compose"](panel, **full_params)
bench_ret, _ = bt.benchmark(panel, fees=fees, slippage=slippage)
bench_cagr = bt._cagr(bench_ret, bt._years(panel.close.index))

def book_excess(score_frame):
    """Run the full slot machinery on any score frame and return its excess CAGR."""
    fake = dict(parts, score_pct=score_frame)
    held = mod["_select"](fake, q_in=full_params["q_in"],
                          q_out=full_params["q_out"],
                          max_hold=full_params["max_hold"],
                          max_positions=full_params["max_positions"],
                          displace=full_params["displace"])["held"]
    pf = bt.run(panel,
                {"weights": mod["_weights"](held, full_params["max_positions"])},
                fees=fees, slippage=slippage, init_cash=settings.backtest.init_cash)
    return bt.portfolio_stats(pf)["cagr"] - bench_cagr

rng = np.random.default_rng(7)
sims = np.array([
    book_excess(pd.DataFrame(rng.random(parts["score_pct"].shape),
                             index=parts["score_pct"].index,
                             columns=parts["score_pct"].columns)
                .where(parts["eligible"]).rank(axis=1, pct=True))
    for _ in range(N_RANDOM_SIMS)])
real = bt.evaluate(strat, panel, picked, settings=settings)["excess_cagr"]
p95 = np.percentile(sims, 95)
z = (real - sims.mean()) / sims.std()
print(f"random books: excess {sims.mean():+.2%}/yr (sd {sims.std():.2%}, p95 {p95:+.2%})")
print(f"the strategy: excess {real:+.2%}/yr  ->  z = {z:+.2f}, "
      f"beats {(sims < real).mean():.0%} of random draws")
print(f"--> {'PASS' if real > p95 else 'WEAK'} (want > p95). This isolates the SIGNAL: "
      "everything else about the two books is identical.")


### 10g · Untouched holdout (open once)

2025–2026 was never seen by the config, the sweep, or the optimizer. Score the
pick on it exactly once. If it disappoints, that is information — do not tune
against it (that would turn the holdout into training data).

In [ ]:
full = load_panel(engine, syms, d["bar"], d["start"], None,
                  session=d.get("session", "regular"), exchanges=settings.exchanges)
features.attach_features(full, engine, d["features"], settings=settings)
holdout = slice_panel(full, "2025-01-01", str(full.close.index[-1].date()))
pf_hold = bt.run_strategy(strat, holdout, params=picked, settings=settings)  # run once, reuse below
st = bt.portfolio_stats(pf_hold)
print(f"HOLDOUT {holdout.close.index[0].date()} → {holdout.close.index[-1].date()} "
      f"({holdout.close.shape[0]} bars):")
for k, v in st.items():
    print(f"  {k:16s} = {v:.4f}")
plots.plot_equity(bt.equity_curve(pf_hold)).show()

#### The holdout book, name by name

`plot_equity` above shows *how* the holdout book did — one aggregate curve. It does
not show *which names* did it. This strategy holds a rotating basket of up to
`max_positions` names, so the question that matters on the holdout is per-name: who
was held, when was each one bought and sold, how much of the book did each occupy, and
what did each contribute.

`strategies/informed_flow/plots.py` — a portfolio-mode-generic figure that lives
beside `strategy.py`, not in `trading_simple.plots`, because "held vs out of the
book" only means something for a rotating multi-name portfolio — draws exactly that
in four x-aligned panels:

1. **Close price** of every name the book ever held: faded across the whole window,
   **bold over the spans it was actually in the portfolio** (read from
   `bt.trade_records`, so these are the real fills), with `▲` a buy and `▼` a sell.
   Each name is held in one go — bought once, then sold whole when its rank falls out,
   never re-trimmed mid-hold (the engine masks orders per name, so a held position is
   left to drift while others rotate) — so a continuous position is exactly **one ▲
   and one ▼**. A name that shows two ▼ genuinely left the book and re-entered later.
2. **Portfolio composition** — each held name's **weight** (its position value as a
   share of the whole book) as a stacked area, with **cash** on top so the stack fills
   to 100%; the hover carries the **share count** and **dollar value** behind each
   weight. This is where the drift shows: with no re-trimming, a winner's band swells
   past its `1/max_positions` slot while the cash band breathes with turnover.
3. **Cumulative realized P&L** — one step line per name (color-matched to panel 1),
   the **total** in ink, and cumulative **fees paid** in amber. The per-name lines
   sum to the total by construction, so a single name carrying the book — or fees
   quietly eating the edge — is visible at a glance.
4. **Portfolio value ($)**, the one shared cash account, against its starting capital.


In [ ]:
import importlib.util

# plot_holdings lives in informed_flow's folder and is portfolio-mode generic, so reuse
# it rather than duplicating: load that module by file (the folder is not on sys.path).
_hp = strategies.load("informed_flow").dir / "plots.py"
_spec = importlib.util.spec_from_file_location("holdings_plots", _hp)
if_plots = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(if_plots)

if_plots.plot_holdings(holdout, pf_hold).show()


## Read the result honestly

- The **holdout** and **walk-forward** Sharpes are the real numbers; the
  in-sample and picked Sharpes are not.
- Judge breadth on **`pct_profitable`**, not one big winner, and always read
  **`n_trades`** next to any Sharpe.
- The **deflated Sharpe** and **random baseline** guard against a lucky pick
  from many trials; the **cost stress** guards against a cost-hurdle mirage.
- ~5 years is a short sample — a strong single number here is weaker evidence
  than the same number would be over 15 years. Prefer consistency across §10e
  to a high headline.
- The yfinance ratings sleeve is survivor-only; insider (SEC) and prices are
  not. Keep claims to this liquid universe.